# 🧠 The Universal Pandas Problem-Solving Framework: Complete Substep Breakdown

Here is a detailed, intuitive breakdown of every single substep in the **4-Step Reverse Engineering Framework**, explaining what it means, why it matters, and how you apply it.

---

Always work backwards from what the stakeholder wants to see:

<pre style="background: transparent !important; background-color: transparent !important; border: none !important; font-family: 'Courier New', Courier, monospace; font-size: 13px; line-height: 1.25; color: inherit; padding: 0; margin: 15px 0;">
┌────────────────────────────────────────────────────────┐
│ STEP 1: TARGET OUTPUT (The Destination)                │
│ • What columns and metrics are required?               │
│ • What does exactly 1 row represent (Grain)?           │
│ • What is the final layout? (Wide matrix vs Long table)│
└──────────────────────────┬─────────────────────────────┘
                           ▼
┌────────────────────────────────────────────────────────┐
│ STEP 2: RAW DATA AUDIT (The Starting Point)            │
│ • Which datasets contain the raw fields?               │
│ • Identify quirks: Nulls, string types, join keys      │
└──────────────────────────┬─────────────────────────────┘
                           ▼
┌────────────────────────────────────────────────────────┐
│ STEP 3: BACKWARD TRANSFORMATION PIPELINE (The Bridge)  │
│ • Ingest & Cast ➔ Filter Early ➔ Merge ➔ Group/Agg ➔   │
│   Derive Ratios ➔ Reshape                              │
│ • Vectorized and clean method-chaining                 │
└──────────────────────────┬─────────────────────────────┘
                           ▼
┌────────────────────────────────────────────────────────┐
│ STEP 4: SANITY AUDIT & VERIFICATION (The Guarantee)    │
│ • Assert unique rows, valid metric bounds (0-100%)     │
│ • Guard against 0-division and join duplicates         │
└────────────────────────────────────────────────────────┘
</pre>

---

# 🎯 STEP 1: TARGET OUTPUT (The Destination)
> **Core Idea:** You cannot build a bridge if you don't know where the other side of the river is. Before writing any code, define the exact deliverable.

### 🔹 Substep 1.1: Target Schema (Required Columns & Metrics)
* **What it means:** List the exact names and types of columns the business stakeholder wants to see.
* **How to do it:** Read the prompt and list them out:
  - Base Dimensions: `['region', 'card_type']`
  - Aggregated Metrics: `['total_transactions', 'fraud_transactions']`
  - Derived Ratios: `['fraud_rate_pct']`
* **Why it matters:** Prevents you from calculating useless columns or forgetting a critical requested metric.

---

### 🔹 Substep 1.2: Target Grain (Level of Detail)
* **What it means:** Ask yourself: *"What does **exactly ONE row** in the final table represent?"*
* **How to do it:**
  - If 1 row = 1 Customer $\rightarrow$ You will need `groupby('customer_id')` or a customer-level table.
  - If 1 row = 1 Region $\times$ Card Type $\rightarrow$ You will need `groupby(['region', 'card_type'])`.
  - If 1 row = 1 Day $\rightarrow$ You will need `resample('D')` or `pd.Grouper(freq='D')`.
* **Why it matters:** **Grain is the #1 concept in data analysis.** If you get the grain wrong, your entire calculation will be duplicated or miscalculated.

---

### 🔹 Substep 1.3: Target Visual Layout (Shape)
* **What it means:** Determine how the data should be structured visually.
* **The 3 Shapes:**
  1. **Flat / Tabular:** Standard rows and columns (default from `groupby`).
  2. **Wide Matrix (Cross-tab):** One dimension on rows, one on columns (created via `.pivot()` or `.unstack()`). E.g., regions as rows, card types as columns.
  3. **Long / Tidy:** Key-value pairs `[id_vars, metric_name, metric_value]` (created via `pd.melt()`) for feeding into BI charting tools like Tableau/PowerBI.

---
---

# 🔍 STEP 2: RAW DATA AUDIT (The Starting Point)
> **Core Idea:** Inspect your raw ingredients to see what cleaning or merging is required before you can perform math.

### 🔹 Substep 2.1: Source Mapping
* **What it means:** Map each column needed in Step 1 back to its raw CSV file.
* **How to do it:**
  - `amount`, `is_fraud`, `card_type` $\rightarrow$ found in `raw_transactions.csv`
  - `account_tier`, `kyc_status` $\rightarrow$ found in `customers.csv`
* **Decision:** If fields come from 2 different CSVs, you know you **must perform a merge/join**.

---

### 🔹 Substep 2.2: Data Cleanliness & Type Auditing
* **What it means:** Inspect raw data for hidden bugs that break calculations silently.
* **What to check:**
  1. **Dates stored as strings:** `'2026-01-01'` cannot do time math $\rightarrow$ convert with `pd.to_datetime()`.
  2. **Whitespace in categories:** `'Visa '` $
eq$ `'Visa'` $
ightarrow$ strip with `.str.strip()`.
  3. **Null values in filter keys:** `status.isna()` $
ightarrow$ handle before filtering.

---

### 🔹 Substep 2.3: Join Keys & Cardinality Check
* **What it means:** Check if the joining column (e.g. `customer_id`) has duplicate rows in the dimension table.
* **Why it matters:** If `customers.csv` has duplicate `customer_id`s, a `left_join` will **silently duplicate transaction rows**, artificially inflating all your sums and counts!

---
---

# 🛠️ STEP 3: BACKWARD TRANSFORMATION PIPELINE (The Bridge)
> **Core Idea:** Transform raw data into the final target output using an orderly, linear execution chain.

Follow this exact 6-step transformation recipe:

**`1. Ingest & Cast ➔ 2. Filter Early ➔ 3. Merge ➔ 4. Group/Aggregate ➔ 5. Derive Ratios ➔ 6. Reshape`**

### 🔹 Substep 3.1: Ingest & Type-Cast
Load the CSVs and convert types right away:

In [ ]:
import pandas as pd
import numpy as np

# Load raw datasets
raw_transactions = pd.read_csv('data/raw_transactions.csv')
customers = pd.read_csv('data/customers.csv')

# Type cast timestamps immediately
raw_transactions['tx_date'] = pd.to_datetime(raw_transactions['created_at'])
raw_transactions.head(2)

### 🔹 Substep 3.2: Filter Early
Filter out irrelevant records before joining/grouping (makes operations 10x faster):

In [ ]:
active_df = raw_transactions[raw_transactions['status'] == 'APPROVED'].copy()
active_df.head(2)

### 🔹 Substep 3.3: Merge / Relational Joins
Combine tables if required:

In [ ]:
merged_df = active_df.merge(customers, on='customer_id', how='left')
merged_df.head(2)

### 🔹 Substep 3.4: Group & Aggregate (Match the Target Grain)
Aggregate columns to reach the exact grain chosen in Step 1.2:

In [ ]:
summary = merged_df.groupby(['region', 'card_type']).agg(
    total_volume=('amount', 'sum'),
    total_tx=('transaction_id', 'count'),
    fraud_tx=('is_fraud', 'sum')
).reset_index()
summary.head()

### 🔹 Substep 3.5: Derive Ratios & KPIs (Defensive Math)
Calculate rates, percentages, and averages. Always protect against dividing by zero:

In [ ]:
summary['fraud_rate_pct'] = (summary['fraud_tx'] / summary['total_tx'] * 100).fillna(0.0)
summary.head()

### 🔹 Substep 3.6: Reshape & Sort
Reorder or reshape if a wide matrix or long format was requested:

In [ ]:
# Wide matrix view
matrix = summary.pivot(index='region', columns='card_type', values='fraud_rate_pct')
print(matrix)

---
### ⚡ Alternative Approach: Fast 2-Block Interview Method Chain

> **Core Idea:** In live 20-30 minute coding interviews, you can also solve Step 3 by collapsing the 6 substeps into **2 high-speed blocks** to eliminate repetitive temporary variables without losing any readability:

<pre style="background: transparent !important; background-color: transparent !important; border: none !important; font-family: 'Courier New', Courier, monospace; font-size: 13px; line-height: 1.25; color: inherit; padding: 0; margin: 15px 0;">
┌────────────────────────────────────────────────────────┐
│ 📦 BLOCK 1: INGEST & FILTER (Substeps 3.1 + 3.2)       │
│ Load and filter records in 1–2 lines using .query()    │
├────────────────────────────────────────────────────────┤
│ ⛓️ BLOCK 2: PIPELINE CHAIN (Substeps 3.3 to 3.6)       │
│ Merge ➔ Group/Aggregate ➔ Derive/Round ➔ Sort in ONE   │
│ continuous method chain                                │
└────────────────────────────────────────────────────────┘
</pre>

#### 🔍 Which Substeps Get Combined?
1. **Combine 3.1 & 3.2 (Ingest & Filter Early):** Load the raw file and apply your row filter immediately (e.g. `pd.read_csv(...).query("status == 'Approved'")` or `[mask]`).
2. **Combine 3.3, 3.4, 3.5, 3.6 (Pipeline Chain):** Start with the filtered table, `.merge()` dimension tables, `.groupby().agg()` to target grain, compute KPIs with `.assign(metric=lambda d: ...)`, `.round(2)`, and `.sort_values()` in a single parenthesized chain.

#### 🎯 Can this be used for all types & levels of questions?
**Yes, 100%.** Here is how the exact same 2-block structure handles every level from Junior to Principal:

| Difficulty Level | How the 2-Block Combination Works |
| :--- | :--- |
| 🟢 **Junior / Entry Level**<br>*(Single table aggregations)* | **Block 1:** Load single CSV.<br>**Block 2:** `.groupby().agg().sort_values()` *(Takes 60 seconds to write)*. |
| 🟡 **Mid-Level**<br>*(2-Table Merges, Anti-Joins, Ratios)* | **Block 1:** Load 2 CSVs with `.query()` filters.<br>**Block 2:** `.merge().groupby().agg().assign(ratio=...).sort_values()` *(Takes 3 mins)*. |
| 🔴 **Senior / Staff Level**<br>*(Multi-Format JSON/TSV, Window Functions)* | **Block 1:** Load JSON / TSV with `json_normalize()`.<br>**Block 2:** Pre-aggregate child table ➔ `.merge()` ➔ `.groupby()` ➔ `.rolling()` ➔ `.sort_values()` *(Takes 5 mins)*. |

#### 🎯 Why interviewers love this pattern:
* **Zero Temporary Clutter:** It doesn't pollute the notebook with 5 unnecessary temporary DataFrames (`df1`, `df2`, `df_temp`).
* **SQL-Like Readability:** Senior engineers read it naturally like a SQL query (`SELECT` ➔ `FROM` ➔ `WHERE` ➔ `GROUP BY` ➔ `ORDER BY`).
* **Speed:** It cuts your coding time in half, giving you extra minutes for assertions and discussion.

In [ ]:
# ⚡ High-Speed 2-Block Live Interview Code Pattern (Alternative Approach):

# BLOCK 1: Ingest & filter early (Substeps 3.1 + 3.2)
tx = pd.read_csv('data/raw_transactions.csv').query("transaction_status == 'Completed'")
cust = pd.read_csv('data/customers.csv')

# BLOCK 2: Pipeline chain: Merge -> Group -> Derive -> Round -> Sort (Substeps 3.3 to 3.6)
fast_summary = (
    tx.merge(cust, on='customer_id', how='inner')
    .groupby('account_tier', as_index=False)
    .agg(
        total_customers=('customer_id', 'nunique'),
        total_volume=('transaction_amount', 'sum'),
        fraud_volume=('is_fraud', 'sum')
    )
    .assign(fraud_rate_pct=lambda d: (d['fraud_volume'] / d['total_volume'] * 100).round(2))
    .round(2)
    .sort_values(by='total_volume', ascending=False)
    .reset_index(drop=True)
)
fast_summary.head(3)

---
---

# 🛡️ STEP 4: SANITY AUDIT & VERIFICATION (The Guarantee)
> **Core Idea:** Don't just assume your code worked—write 3 quick checks to prove that the result is 100% correct.

### 🔹 Substep 4.1: Target Grain Uniqueness Check
Check: Did your groupby produce any duplicate primary keys?

In [ ]:
assert not summary.duplicated(subset=['region', 'card_type']).any(), "Error: Duplicate grain found!"
print("✅ Substep 4.1 Passed: No duplicate grain keys!")

### 🔹 Substep 4.2: Range & Boundary Invariants
Check: Are rates logically valid (e.g., percentages must be between 0% and 100%)?

In [ ]:
assert summary['fraud_rate_pct'].between(0, 100).all(), "Error: Rate out of bounds!"
print("✅ Substep 4.2 Passed: All rates within valid 0-100% bounds!")

### 🔹 Substep 4.3: Missing Value Audit
Check: Did any calculations accidentally introduce unexpected NaN or inf values?

In [ ]:
assert summary[['total_volume', 'total_tx']].isna().sum().sum() == 0, "Error: Unexpected NaNs!"
print("✅ Substep 4.3 Passed: Zero unexpected NaNs in core metrics!")

---
---

# 💡 Quick Summary

| Step | Mental Question | Action |
| :--- | :--- | :--- |
| **Step 1 (Output)** | *"What does the final table look like?"* | Define columns, 1-row grain, layout. |
| **Step 2 (Audit)** | *"What raw tables and quirks do I have?"* | Identify sources, data types, nulls. |
| **Step 3 (Pipeline)** | *"How do I connect raw $\rightarrow$ final?"* | Ingest & Cast ➔ Filter Early ➔ Merge ➔ Group/Agg ➔ Derive ➔ Reshape. |
| **Step 4 (Verify)** | *"How do I prove my answer is correct?"* | Check duplicates, valid bounds, null counts. |